#### 1. Traffic Exhaust Emission

* Aim:To visualise emission (kg) of each grid into jpg and google map background

In [1]:
"""
Emission Heatmap Visualiser — Chandivali & Sion
================================================
Generates:
  - chandivali_emission_heatmap.jpg
  - sion_emission_heatmap.jpg
  - emission_heatmaps_combined.html   (both regions, interactive, transparent overlay on Google Maps)

Requirements:
    pip install pandas openpyxl matplotlib contextily folium branca numpy shapely

Edit the three path variables below before running.
"""

import os
import warnings
warnings.filterwarnings("ignore")

# ── USER-EDITABLE PATHS ────────────────────────────────────────────────────────
EXCEL_PATH   = r"D:/LAMP/Traffic_Survey/Road_Type_Length_Calculations/Emission_Data/grid_details_consolidated_Emissions.xlsx"   # input Excel
OUTPUT_DIR   = r"C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions"                                    # folder for outputs
GOOGLE_MAPS_API_KEY = ""   # optional: paste your key for richer basemap in HTML; leave "" to use free tiles
# ──────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from matplotlib.colorbar import ColorbarBase
import contextily as ctx
import geopandas as gpd
from shapely.geometry import box
import folium
from folium.plugins import FloatImage
import branca.colormap as bcm
import branca
import base64
from io import BytesIO

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load data ──────────────────────────────────────────────────────────────────
df_c = pd.read_excel(EXCEL_PATH, sheet_name="CHANDIVALI")
df_s = pd.read_excel(EXCEL_PATH, sheet_name="SION")

DATASETS = {
    "Chandivali": df_c,
    "Sion":       df_s,
}

# ── Colour palette (yellow → orange → red → dark-red) ─────────────────────────
CMAP_NAME = "YlOrRd"


# ═══════════════════════════════════════════════════════════════════════════════
# PART 1 — JPG (matplotlib + contextily satellite/OSM basemap)
# ═══════════════════════════════════════════════════════════════════════════════

def make_jpg(region_name: str, df: pd.DataFrame, out_path: str):
    """Render grids as transparent coloured rectangles over a map tile basemap."""

    emission_col = "Emission (kg)"
    vmin, vmax = df[emission_col].min(), df[emission_col].max()
    norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap  = cm.get_cmap(CMAP_NAME)

    # Build GeoDataFrame (WGS-84 → Web Mercator for contextily)
    geoms = [
        box(row["EXT_MIN_X"], row["EXT_MIN_Y"], row["EXT_MAX_X"], row["EXT_MAX_Y"])
        for _, row in df.iterrows()
    ]
    gdf = gpd.GeoDataFrame(df, geometry=geoms, crs="EPSG:4326").to_crs("EPSG:3857")

    fig, ax = plt.subplots(figsize=(12, 11))

    # Draw each grid cell
    for _, row in gdf.iterrows():
        xmin, ymin, xmax, ymax = row.geometry.bounds
        colour = cmap(norm(row[emission_col]))
        rect = Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=0.6,
            edgecolor="white",
            facecolor=(*colour[:3], 0.55),   # 55 % opacity
        )
        ax.add_patch(rect)

    # Fit axes to data extent with a small margin
    xmin_all = gdf.geometry.bounds["minx"].min()
    xmax_all = gdf.geometry.bounds["maxx"].max()
    ymin_all = gdf.geometry.bounds["miny"].min()
    ymax_all = gdf.geometry.bounds["maxy"].max()
    margin_x = (xmax_all - xmin_all) * 0.05
    margin_y = (ymax_all - ymin_all) * 0.05
    ax.set_xlim(xmin_all - margin_x, xmax_all + margin_x)
    ax.set_ylim(ymin_all - margin_y, ymax_all + margin_y)

    # Google Maps-style basemap (OpenStreetMap — no API key needed)
    try:
        ctx.add_basemap(ax, crs=gdf.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom="auto")
    except Exception:
        try:
            ctx.add_basemap(ax, crs=gdf.crs, zoom="auto")
        except Exception as e:
            print(f"  [Warning] basemap could not be downloaded: {e}")

    # Colour-bar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, aspect=30)
    cbar.set_label("Emission Intensity (kg/day)", fontsize=12, labelpad=10)
    cbar.ax.tick_params(labelsize=10)

    ax.set_title(f"{region_name} — Emission Intensity Heatmap\n(500 m × 500 m grids)",
                 fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Longitude (Web Mercator)", fontsize=9)
    ax.set_ylabel("Latitude (Web Mercator)", fontsize=9)
    ax.tick_params(labelsize=8)

    plt.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved → {out_path}")


print("Generating JPG files …")
for name, df in DATASETS.items():
    out = os.path.join(OUTPUT_DIR, f"{name.lower()}_emission_heatmap.jpg")
    make_jpg(name, df, out)


# ═══════════════════════════════════════════════════════════════════════════════
# PART 2 — HTML (Folium interactive map, transparent grid overlay)
# ═══════════════════════════════════════════════════════════════════════════════

def make_colormap(vmin, vmax, cmap_name=CMAP_NAME):
    """Return a branca StepColormap with 10 steps."""
    mpl_cmap = cm.get_cmap(cmap_name)
    steps    = 10
    colors   = [mcolors.to_hex(mpl_cmap(i / (steps - 1))) for i in range(steps)]
    cmap_b   = bcm.LinearColormap(
        colors=colors,
        vmin=vmin, vmax=vmax,
        caption="Emission Intensity (kg/day)",
    )
    return cmap_b


def add_region_layer(fmap, df, region_name, opacity=0.55):
    emission_col = "Emission (kg)"
    vmin, vmax   = df[emission_col].min(), df[emission_col].max()
    cmap_b       = make_colormap(vmin, vmax)

    fg = folium.FeatureGroup(name=region_name, show=True)

    for _, row in df.iterrows():
        colour = cmap_b(row[emission_col])
        # Convert hex color to rgba string for fill
        r, g, b = mcolors.hex2color(colour)
        fill_opacity = opacity

        folium.Rectangle(
            bounds=[
                [row["EXT_MIN_Y"], row["EXT_MIN_X"]],
                [row["EXT_MAX_Y"], row["EXT_MAX_X"]],
            ],
            color="white",
            weight=0.7,
            fill=True,
            fill_color=colour,
            fill_opacity=fill_opacity,
            tooltip=folium.Tooltip(
                f"<b>{region_name} Grid {int(row['CODE'])}</b><br>"
                f"Emission: <b>{row[emission_col]:,.1f} kg/day</b>",
                sticky=True,
            ),
        ).add_to(fg)

    fg.add_to(fmap)

    # Add colormap legend — position below previous one if multiple
    cmap_b.add_to(fmap)
    return cmap_b


print("\nGenerating combined HTML …")

# Centre map between both regions
all_lats = pd.concat([df_c["EXT_MIN_Y"], df_c["EXT_MAX_Y"],
                       df_s["EXT_MIN_Y"], df_s["EXT_MAX_Y"]])
all_lons = pd.concat([df_c["EXT_MIN_X"], df_c["EXT_MAX_X"],
                       df_s["EXT_MIN_X"], df_s["EXT_MAX_X"]])
center   = [all_lats.mean(), all_lons.mean()]

fmap = folium.Map(
    location=center,
    zoom_start=14,
    tiles=None,          # we add tiles manually so we can label them
)

# ── Tile layers ────────────────────────────────────────────────────────────────
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap",
    max_zoom=19,
).add_to(fmap)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="© Google Maps",
    name="Google Maps (Road)",
    max_zoom=20,
).add_to(fmap)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="© Google Satellite",
    name="Google Satellite",
    max_zoom=20,
).add_to(fmap)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    attr="© Google Hybrid",
    name="Google Hybrid",
    max_zoom=20,
).add_to(fmap)

# ── Data layers ────────────────────────────────────────────────────────────────
add_region_layer(fmap, df_c, "Chandivali")
add_region_layer(fmap, df_s, "Sion")

# ── Layer control + full-screen ───────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(fmap)

# ── Custom title box ──────────────────────────────────────────────────────────
title_html = """
<div style="
    position: fixed;
    top: 10px; left: 50%; transform: translateX(-50%);
    z-index: 9999;
    background: rgba(255,255,255,0.88);
    padding: 8px 18px;
    border-radius: 6px;
    font-family: Arial, sans-serif;
    font-size: 15px;
    font-weight: bold;
    box-shadow: 0 2px 8px rgba(0,0,0,0.25);
    pointer-events: none;
">
    Mumbai Emission Intensity Heatmap — Chandivali &amp; Sion (500 m × 500 m grids)
</div>
"""
fmap.get_root().html.add_child(folium.Element(title_html))

# ── Info box (bottom-left) ─────────────────────────────────────────────────────
info_html = """
<div style="
    position: fixed;
    bottom: 30px; left: 12px;
    z-index: 9999;
    background: rgba(255,255,255,0.82);
    padding: 7px 12px;
    border-radius: 5px;
    font-family: Arial, sans-serif;
    font-size: 12px;
    box-shadow: 0 1px 5px rgba(0,0,0,0.2);
    line-height: 1.6;
    pointer-events: none;
">
    <b>How to use:</b><br>
    • Hover over a grid for emission value<br>
    • Toggle regions via layer panel (top-right)<br>
    • Switch basemap (OSM / Google Maps / Satellite)<br>
    • Scroll to zoom, drag to pan
</div>
"""
fmap.get_root().html.add_child(folium.Element(info_html))

html_path = os.path.join(OUTPUT_DIR, "emission_heatmaps_combined.html")
fmap.save(html_path)
print(f"  Saved → {html_path}")

print("\n✅  All files generated successfully.")
print(f"   • {os.path.join(OUTPUT_DIR, 'chandivali_emission_heatmap.jpg')}")
print(f"   • {os.path.join(OUTPUT_DIR, 'sion_emission_heatmap.jpg')}")
print(f"   • {html_path}")

Generating JPG files …
  Saved → C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\chandivali_emission_heatmap.jpg
  Saved → C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\sion_emission_heatmap.jpg

Generating combined HTML …
  Saved → C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\emission_heatmaps_combined.html

✅  All files generated successfully.
   • C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\chandivali_emission_heatmap.jpg
   • C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\sion_emission_heatmap.jpg
   • C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\emission_heatmaps_combined.html


#### 2. Sweeping

In [22]:
"""
Emission Heatmap Visualiser — Chandivali & Sion
================================================
Generates:
  - chandivali_emission_heatmap.jpg
  - sion_emission_heatmap.jpg
  - emission_heatmaps_combined.html   (both regions, interactive, transparent overlay on Google Maps)

Requirements:
    pip install pandas openpyxl matplotlib contextily folium branca numpy shapely

Edit the three path variables below before running.
"""

import os
import warnings
warnings.filterwarnings("ignore")

# ── USER-EDITABLE PATHS ────────────────────────────────────────────────────────
EXCEL_PATH   = r"D:/LAMP/Traffic_Survey/Road_Type_Length_Calculations/grids_saptarshi/Sweeping_Heatmap_Grid_Details_PM10.xlsx"   # input Excel
OUTPUT_DIR   = r"C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions"                                    # folder for outputs
GOOGLE_MAPS_API_KEY = ""   # optional: paste your key for richer basemap in HTML; leave "" to use free tiles
# ──────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from matplotlib.colorbar import ColorbarBase
import contextily as ctx
import geopandas as gpd
from shapely.geometry import box
import folium
from folium.plugins import FloatImage
import branca.colormap as bcm
import branca
import base64
from io import BytesIO

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load data ──────────────────────────────────────────────────────────────────
df_c = pd.read_excel(EXCEL_PATH, sheet_name="CHANDIVALI")
df_s = pd.read_excel(EXCEL_PATH, sheet_name="SION")

DATASETS = {
    "Chandivali": df_c,
    "Sion":       df_s,
}

# ── Colour palette (yellow → orange → red → dark-red) ─────────────────────────
CMAP_NAME = "YlOrRd"


# ═══════════════════════════════════════════════════════════════════════════════
# PART 1 — JPG (matplotlib + contextily satellite/OSM basemap)
# ═══════════════════════════════════════════════════════════════════════════════

def make_jpg(region_name: str, df: pd.DataFrame, out_path: str):
    """Render grids as transparent coloured rectangles over a map tile basemap."""

    emission_col = "Sweeping Emission Kg/day"
    vmin, vmax = df[emission_col].min(), df[emission_col].max()
    norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap  = cm.get_cmap(CMAP_NAME)

    # Build GeoDataFrame (WGS-84 → Web Mercator for contextily)
    geoms = [
        box(row["EXT_MIN_X"], row["EXT_MIN_Y"], row["EXT_MAX_X"], row["EXT_MAX_Y"])
        for _, row in df.iterrows()
    ]
    gdf = gpd.GeoDataFrame(df, geometry=geoms, crs="EPSG:4326").to_crs("EPSG:3857")

    fig, ax = plt.subplots(figsize=(12, 11))

    # Draw each grid cell
    for _, row in gdf.iterrows():
        xmin, ymin, xmax, ymax = row.geometry.bounds
        colour = cmap(norm(row[emission_col]))
        rect = Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=0.6,
            edgecolor="white",
            facecolor=(*colour[:3], 0.55),   # 55 % opacity
        )
        ax.add_patch(rect)

    # Fit axes to data extent with a small margin
    xmin_all = gdf.geometry.bounds["minx"].min()
    xmax_all = gdf.geometry.bounds["maxx"].max()
    ymin_all = gdf.geometry.bounds["miny"].min()
    ymax_all = gdf.geometry.bounds["maxy"].max()
    margin_x = (xmax_all - xmin_all) * 0.05
    margin_y = (ymax_all - ymin_all) * 0.05
    ax.set_xlim(xmin_all - margin_x, xmax_all + margin_x)
    ax.set_ylim(ymin_all - margin_y, ymax_all + margin_y)

    # Google Maps-style basemap (OpenStreetMap — no API key needed)
    try:
        ctx.add_basemap(ax, crs=gdf.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom="auto")
    except Exception:
        try:
            ctx.add_basemap(ax, crs=gdf.crs, zoom="auto")
        except Exception as e:
            print(f"  [Warning] basemap could not be downloaded: {e}")

    # Colour-bar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, aspect=30)
    cbar.set_label("Emission (kg/day)", fontsize=12, labelpad=10)
    cbar.ax.tick_params(labelsize=10)

    ax.set_title(f"{region_name} — Emission Intensity Heatmap\n(500 m × 500 m grids)",
                 fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Longitude (Web Mercator)", fontsize=9)
    ax.set_ylabel("Latitude (Web Mercator)", fontsize=9)
    ax.tick_params(labelsize=8)

    plt.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved → {out_path}")


print("Generating JPG files …")
for name, df in DATASETS.items():
    out = os.path.join(OUTPUT_DIR, f"{name.lower()}_sweeping_heatmap_PM2.5.jpg") # emission_heatmap
    make_jpg(name, df, out)


# ═══════════════════════════════════════════════════════════════════════════════
# PART 2 — HTML (Folium interactive map, transparent grid overlay)
# ═══════════════════════════════════════════════════════════════════════════════

def make_colormap(vmin, vmax, cmap_name=CMAP_NAME):
    """Return a branca StepColormap with 10 steps."""
    mpl_cmap = cm.get_cmap(cmap_name)
    steps    = 10
    colors   = [mcolors.to_hex(mpl_cmap(i / (steps - 1))) for i in range(steps)]
    cmap_b   = bcm.LinearColormap(
        colors=colors,
        vmin=vmin, vmax=vmax,
        caption="Sweeping Emission Kg/day",
    )
    return cmap_b


def add_region_layer(fmap, df, region_name, opacity=0.55):
    emission_col = "Emission (kg)"
    vmin, vmax   = df[emission_col].min(), df[emission_col].max()
    cmap_b       = make_colormap(vmin, vmax)

    fg = folium.FeatureGroup(name=region_name, show=True)

    for _, row in df.iterrows():
        colour = cmap_b(row[emission_col])
        # Convert hex color to rgba string for fill
        r, g, b = mcolors.hex2color(colour)
        fill_opacity = opacity

        folium.Rectangle(
            bounds=[
                [row["EXT_MIN_Y"], row["EXT_MIN_X"]],
                [row["EXT_MAX_Y"], row["EXT_MAX_X"]],
            ],
            color="white",
            weight=0.7,
            fill=True,
            fill_color=colour,
            fill_opacity=fill_opacity,
            tooltip=folium.Tooltip(
                f"<b>{region_name} Grid {int(row['CODE'])}</b><br>"
                f"Emission: <b>{row[emission_col]:,.1f} kg/day</b>",
                sticky=True,
            ),
        ).add_to(fg)

    fg.add_to(fmap)

    # Add colormap legend — position below previous one if multiple
    cmap_b.add_to(fmap)
    return cmap_b


print("\nGenerating combined HTML …")

# Centre map between both regions
all_lats = pd.concat([df_c["EXT_MIN_Y"], df_c["EXT_MAX_Y"],
                       df_s["EXT_MIN_Y"], df_s["EXT_MAX_Y"]])
all_lons = pd.concat([df_c["EXT_MIN_X"], df_c["EXT_MAX_X"],
                       df_s["EXT_MIN_X"], df_s["EXT_MAX_X"]])
center   = [all_lats.mean(), all_lons.mean()]

fmap = folium.Map(
    location=center,
    zoom_start=14,
    tiles=None,          # we add tiles manually so we can label them
)

# ── Tile layers ────────────────────────────────────────────────────────────────
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap",
    max_zoom=19,
).add_to(fmap)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="© Google Maps",
    name="Google Maps (Road)",
    max_zoom=20,
).add_to(fmap)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="© Google Satellite",
    name="Google Satellite",
    max_zoom=20,
).add_to(fmap)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    attr="© Google Hybrid",
    name="Google Hybrid",
    max_zoom=20,
).add_to(fmap)

# ── Data layers ────────────────────────────────────────────────────────────────
add_region_layer(fmap, df_c, "Chandivali")
add_region_layer(fmap, df_s, "Sion")

# ── Layer control + full-screen ───────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(fmap)

# ── Custom title box ──────────────────────────────────────────────────────────
title_html = """
<div style="
    position: fixed;
    top: 10px; left: 50%; transform: translateX(-50%);
    z-index: 9999;
    background: rgba(255,255,255,0.88);
    padding: 8px 18px;
    border-radius: 6px;
    font-family: Arial, sans-serif;
    font-size: 15px;
    font-weight: bold;
    box-shadow: 0 2px 8px rgba(0,0,0,0.25);
    pointer-events: none;
">
    Mumbai Emission Intensity Heatmap — Chandivali &amp; Sion (500 m × 500 m grids)
</div>
"""
fmap.get_root().html.add_child(folium.Element(title_html))

# ── Info box (bottom-left) ─────────────────────────────────────────────────────
info_html = """
<div style="
    position: fixed;
    bottom: 30px; left: 12px;
    z-index: 9999;
    background: rgba(255,255,255,0.82);
    padding: 7px 12px;
    border-radius: 5px;
    font-family: Arial, sans-serif;
    font-size: 12px;
    box-shadow: 0 1px 5px rgba(0,0,0,0.2);
    line-height: 1.6;
    pointer-events: none;
">
    <b>How to use:</b><br>
    • Hover over a grid for emission value<br>
    • Toggle regions via layer panel (top-right)<br>
    • Switch basemap (OSM / Google Maps / Satellite)<br>
    • Scroll to zoom, drag to pan
</div>
"""
fmap.get_root().html.add_child(folium.Element(info_html))

html_path = os.path.join(OUTPUT_DIR, "sweeping_heatmaps_combined_PM10.html")
fmap.save(html_path)
print(f"  Saved → {html_path}")

print("\n✅  All files generated successfully.")
print(f"   • {os.path.join(OUTPUT_DIR, 'chandivali_sweeping_heatmap_PM10.jpg')}")
print(f"   • {os.path.join(OUTPUT_DIR, 'sion_sweeping_heatmap_PM10.jpg')}")
print(f"   • {html_path}")

Generating JPG files …
  Saved → C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\chandivali_sweeping_heatmap_PM2.5.jpg
  Saved → C:/Users/Atique/LAMP_Coding/Traffic_Survey/Visulaisations_Emissions\sion_sweeping_heatmap_PM2.5.jpg

Generating combined HTML …


KeyError: 'Emission (kg)'